In [13]:

# %% [markdown]
# # Assignment Five: Unsupervised Learning — Part B
# ## Notebook: customer_segmentation.ipynb
# **Goal:** Implement a complete customer segmentation pipeline using K-Means and an independently researched alternative clustering method.

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [14]:
# 1. LOAD DATASET
# ==========================================
print("=== Step 1: Load Dataset ===")
df = pd.read_csv('raw_wholesale_customers.csv')

print("\n--- First 5 Rows (Head) ---")
print(df.head())

print("\n--- DataFrame Summary Info ---")
print(df.info())

=== Step 1: Load Dataset ===

--- First 5 Rows (Head) ---
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185

--- DataFrame Summary Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 440 entries, 0 to 439
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Channel           440 non-null    int64
 1   Region            440 non-null    int64
 2   Fresh             440 non-null    int64
 3   Milk              440 non-null    int64
 4   Grocery           440 non-null    int64
 5   Frozen      

In [15]:
# 2. SELECT FEATURES + IQR CAP
# ==========================================
print("\n=== Step 2: Select Features + IQR Cap ===")

# Exclude metadata columns 'Channel' and 'Region' from clustering inputs
spend_cols = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
X = df[spend_cols].copy()

# Apply non-destructive IQR capping (k=1.5)
for col in spend_cols:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Clip extreme outlier values instead of dropping observations
    X[col] = X[col].clip(lower_bound, upper_bound)

print("\nFeatures summary description after IQR clipping:")
print(X.describe())


=== Step 2: Select Features + IQR Cap ===

Features summary description after IQR clipping:
              Fresh          Milk      Grocery       Frozen  Detergents_Paper  \
count    440.000000    440.000000    440.00000   440.000000        440.000000   
mean   11357.568182   5048.592045   7236.37500  2507.085795       2392.616477   
std    10211.542235   4386.377073   6596.53308  2408.297738       2940.794090   
min        3.000000     55.000000      3.00000    25.000000          3.000000   
25%     3127.750000   1533.000000   2153.00000   742.250000        256.750000   
50%     8504.000000   3627.000000   4755.50000  1526.000000        816.500000   
75%    16933.750000   7190.250000  10655.75000  3554.250000       3922.000000   
max    37642.750000  15676.125000  23409.87500  7772.250000       9419.875000   

        Delicassen  
count   440.000000  
mean   1266.715341  
std    1083.069792  
min       3.000000  
25%     408.250000  
50%     965.500000  
75%    1820.250000  
max    39

In [16]:
# 3. SCALE FEATURES
# ==========================================
print("\n=== Step 3: Scale Features ===")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nFirst 3 rows of scaled spend features:\n", X_scaled[:3])


=== Step 3: Scale Features ===

First 3 rows of scaled spend features:
 [[ 0.12857261  1.05158597  0.04926747 -0.95324427  0.09579175  0.06589216]
 [-0.42162716  1.08673463  0.35386453 -0.30973493  0.30651872  0.47075856]
 [-0.49064723  0.85804007  0.06793486 -0.04243744  0.38243489  2.46943983]]


In [18]:
# 4. ELBOW METHOD
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
print("\n=== Step 4: Elbow Method ===")

sse = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    km.fit(X_scaled)
    sse.append(km.inertia_)
    print(f"k = {k} | Sum of Squared Errors (SSE / Inertia): {km.inertia_:.4f}")

# Generating and saving the elbow plot
plt.plot(k_range, sse, marker='o', color='b', linestyle='-')
plt.title('Elbow Method for Optimal k Selection', fontsize=12)
plt.xlabel('Number of Clusters (k)', fontsize=10)
plt.ylabel('Sum of Squared Errors (SSE)', fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.savefig('elbow_plot.png')
plt.close()
print("\n[Checkpoint] Elbow diagram saved as 'elbow_plot.png'.")


=== Step 4: Elbow Method ===
k = 1 | Sum of Squared Errors (SSE / Inertia): 2640.0000
k = 2 | Sum of Squared Errors (SSE / Inertia): 1728.1886
k = 3 | Sum of Squared Errors (SSE / Inertia): 1363.4478
k = 4 | Sum of Squared Errors (SSE / Inertia): 1202.4114
k = 5 | Sum of Squared Errors (SSE / Inertia): 1070.1467
k = 6 | Sum of Squared Errors (SSE / Inertia): 964.7603
k = 7 | Sum of Squared Errors (SSE / Inertia): 921.1434
k = 8 | Sum of Squared Errors (SSE / Inertia): 776.6322
k = 9 | Sum of Squared Errors (SSE / Inertia): 726.8756
k = 10 | Sum of Squared Errors (SSE / Inertia): 707.4116

[Checkpoint] Elbow diagram saved as 'elbow_plot.png'.


In [19]:
# 5. TRAIN K-MEANS (REPRODUCE LESSON)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
print("\n=== Step 5: Train K-Means (n_clusters=5) ===")

kmeans_model = KMeans(n_clusters=5, n_init="auto", random_state=42)
df['KMeans_Cluster'] = kmeans_model.fit_predict(X_scaled)

print("\nK-Means Cluster Value Distribution Counts:")
print(df['KMeans_Cluster'].value_counts())


=== Step 5: Train K-Means (n_clusters=5) ===

K-Means Cluster Value Distribution Counts:
KMeans_Cluster
1    191
3     88
0     76
4     60
2     25
Name: count, dtype: int64


In [20]:
# 6. EVALUATE K-MEANS
# ==========================================
print("\n=== Step 6: Evaluate K-Means ===")

sil_km = silhouette_score(X_scaled, df['KMeans_Cluster'])
db_km = davies_bouldin_score(X_scaled, df['KMeans_Cluster'])

print(f"K-Means Silhouette Score     : {sil_km:.4f}  (Higher is better, scale -1 to +1)")
print(f"K-Means Davies-Bouldin Index  : {db_km:.4f}  (Lower is better, scale 0+)")

# Print cluster centers converted back to original spend units
raw_centers = scaler.inverse_transform(kmeans_model.cluster_centers_)
centers_df = pd.DataFrame(raw_centers, columns=spend_cols)
print("\nCluster Centroids in Original Currency/Spend Units:")
print(centers_df)


=== Step 6: Evaluate K-Means ===
K-Means Silhouette Score     : 0.2831  (Higher is better, scale -1 to +1)
K-Means Davies-Bouldin Index  : 1.2701  (Lower is better, scale 0+)

Cluster Centroids in Original Currency/Spend Units:
          Fresh          Milk       Grocery       Frozen  Detergents_Paper  \
0   9202.671053   6833.302632   9104.118421  1326.157895       3280.118421   
1   8376.230366   2150.649215   3160.628272  1646.329843        779.251309   
2  17461.540000  13805.605000  17524.120000  4120.570000       5460.565000   
3  22346.698864   3409.137784   3969.329545  5819.596591        583.068182   
4   4916.983333  10768.854167  18350.133333  1212.366667       7780.018750   

    Delicassen  
0  1871.756579  
1   674.015707  
2  3583.640000  
3  1566.946023  
4   981.366667  


In [21]:
# 7. RESEARCH & TRAIN A SECOND ALGORITHM
# ==========================================
print("\n=== Step 7: Train Researched Second Clustering Algorithm (Hierarchical) ===")

agg_model = AgglomerativeClustering(n_clusters=5, linkage='ward')
df['Agg_Cluster'] = agg_model.fit_predict(X_scaled)

print("Agglomerative Clustering model fitted successfully.")


=== Step 7: Train Researched Second Clustering Algorithm (Hierarchical) ===
Agglomerative Clustering model fitted successfully.


In [22]:
# 8. COMPARE METHODS
# ==========================================
print("\n=== Step 8: Compare Methods ===")

sil_agg = silhouette_score(X_scaled, df['Agg_Cluster'])

print(f"K-Means Silhouette Score               : {sil_km:.4f}")
print(f"Agglomerative Clustering Silhouette Score : {sil_agg:.4f}")

if sil_km > sil_agg:
    print("\n💡 Comparison Note: K-Means produced a higher Silhouette Score, indicating cleaner and better-separated customer partitions for this configuration.")
else:
    print("\n💡 Comparison Note: Agglomerative Hierarchical clustering achieved higher cohesion and separation in this feature space.")


=== Step 8: Compare Methods ===
K-Means Silhouette Score               : 0.2831
Agglomerative Clustering Silhouette Score : 0.2185

💡 Comparison Note: K-Means produced a higher Silhouette Score, indicating cleaner and better-separated customer partitions for this configuration.


In [ ]:
# 9. SANITY CHECK
# ==========================================
print("\n=== Step 9: Sanity Check ===")

# Extract three representative test clients from the dataset
sample_clients = df.head(3)

print("Inspection of chosen sample clients:")
print(sample_clients[['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen', 
                      'Channel', 'Region', 'KMeans_Cluster', 'Agg_Cluster']])